# Fase 2 — Ingestão Incremental em Batch dos Logs da Urna (`fact_urna_logs`)

Visão geral da ingestão incremental em batch dos logs da urna eletrônica (`fact_urna_logs` — 15 colunas 1:1 com `message_text` mascarado para números de 5+ dígitos como `<ID_REDACTED>` e sem `mac_code`) por UF e Turno em Parquet ZSTD particionado (`election_year=2026/turn=T/uf=UF`).

### Objetivos:
- Inspeção autocontida do Central Directory dos pacotes oficiais `bu_imgbu_logjez_rdv_vscmr_2026_{turn}t_{uf}.zip` (suportando tanto arquivo ZIP local em `data/staging/election_year=2026/turn={turn}/uf={uf}/` quanto leitura seletiva via HTTP Range de entradas `o<pleito:5><uf:2><mun:5><zona:4><secao:4>-log.jez` com `MAX_JEZ_FILES_PER_PARTITION`).
- Extração 100% em memória (`io.BytesIO`) de contêineres `.jez` com detecção automática de assinatura mágica 7z/LZMA (`py7zr`, `37 7A BC AF 27 1C`) e ZIP (`zipfile`, `50 4B 03 04`), sem gravar `.jez` ou `logd.dat` descompactados em disco permanente.
- Ordenação determinística de segmentos rotacionados (`logd.dat.2` -> `logd.dat.1` -> `logd.dat`) com numeração sequencial `file_seq` (`Int64`, `1..N`) estritamente monotônica por `.jez` e registro de `source_log_member`.
- Conversão 1:1 das linhas tabuladas (`\t`) de `logd.dat` (UTF-8 com fallback Latin-1) para as 15 colunas de `fact_urna_logs`, aplicando `re.sub(r"\b\d{5,}\b", "<ID_REDACTED>", msg)` em `message_text` e omitindo o 6º campo hexadecimal `mac_code`.
- Escrita atômica e idempotente em Parquet ZSTD (`compression='zstd'`, `compression_level=5`, `use_dictionary=True`) em `data/exports/fact_urna_logs/election_year=2026/turn={turn}/uf={uf}/part-0.parquet`, reconciliação física `processed_jez_count == target_jez_count` e `physical_logd_lines_count == parquet_rows_count`, limpeza imediata de `data/staging/` e atualização atômica de `data/exports/_checkpoints/manifest.json`.

In [1]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

import polars as pl
import pyarrow as pa

for _pkg in ("polars", "pyarrow", "httpx", "py7zr"):
    if importlib.util.find_spec(_pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])

# Parâmetros idempotentes e 100% autocontidos de execução
ELECTION_YEAR = 2026
TARGET_TURNS = [1, 2]
ALL_UFS = [
    "AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO", "MA",
    "MG", "MS", "MT", "PA", "PB", "PE", "PI", "PR", "RJ", "RN",
    "RO", "RR", "RS", "SC", "SE", "SP", "TO", "ZZ"
]
TARGET_UFS = list(ALL_UFS)
MAX_JEZ_FILES_PER_PARTITION: int | None = 5  # 5 por partição para execução rápida no Kernel; None para todos os .jez da UF
SKIP_EXISTING = True
FORCE_REFRESH = False
STAGING_DIR = Path("data/staging")
EXPORTS_DIR = Path("data/exports")
MANIFEST_PATH = Path("data/exports/_checkpoints/manifest.json")

DEFAULT_URL_TEMPLATE = (
    "https://cdn.tse.jus.br/estatistica/sead/eleicoes/eleicoes{election_year}/"
    "arqurnatot/bu_imgbu_logjez_rdv_vscmr_{election_year}_{turn}t_{uf}.zip"
)
DEFAULT_LOG_DOCS_URL = (
    "https://www.tse.jus.br/eleicoes/eleicoes-2026-content/arquivos/"
    "log-do-ecossistema/@@display-file/file/log-do-ecossistema.pdf"
)


def load_urna_logs_source_definitions() -> dict:
    """Retorna as definições autocontidas de totalizacao_arquivos_transmitidos e ecossistema_urna_log_docs."""
    return {
        "url_template": DEFAULT_URL_TEMPLATE,
        "situacao": "Totalizado",
        "target_table": "fact_urna_logs",
        "docs_url": DEFAULT_LOG_DOCS_URL,
        "delimiter": "\t",
    }


SOURCE_DEFS = load_urna_logs_source_definitions()
URL_TEMPLATE = SOURCE_DEFS["url_template"]
REQUIRED_SITUACAO = SOURCE_DEFS["situacao"]
LOG_DOCS_URL = SOURCE_DEFS["docs_url"]

# Schema Polars/PyArrow de 15 colunas para fact_urna_logs
FACT_URNA_LOGS_SCHEMA = {
    "election_year": pl.Int32,
    "turn": pl.Int8,
    "uf": pl.Utf8,
    "cd_pleito": pl.Int64,
    "cd_municipio": pl.Int32,
    "nr_zona": pl.Int32,
    "nr_secao": pl.Int32,
    "source_jez_file": pl.Utf8,
    "source_log_member": pl.Utf8,
    "file_seq": pl.Int64,
    "raw_timestamp": pl.Utf8,
    "log_level": pl.Utf8,
    "urna_serial": pl.Utf8,
    "application": pl.Utf8,
    "message_text": pl.Utf8,
}

FACT_URNA_LOGS_ARROW_SCHEMA = pa.schema([
    ("election_year", pa.int32()),
    ("turn", pa.int8()),
    ("uf", pa.string()),
    ("cd_pleito", pa.int64()),
    ("cd_municipio", pa.int32()),
    ("nr_zona", pa.int32()),
    ("nr_secao", pa.int32()),
    ("source_jez_file", pa.string()),
    ("source_log_member", pa.string()),
    ("file_seq", pa.int64()),
    ("raw_timestamp", pa.string()),
    ("log_level", pa.string()),
    ("urna_serial", pa.string()),
    ("application", pa.string()),
    ("message_text", pa.string()),
])

FACT_URNA_LOGS_COLUMNS = list(FACT_URNA_LOGS_SCHEMA.keys())

print(
    f"Configuração carregada para {SOURCE_DEFS['target_table']} "
    f"(ano={ELECTION_YEAR}, turnos={TARGET_TURNS}, UFs={TARGET_UFS}, "
    f"situacao={REQUIRED_SITUACAO!r}, max_jez={MAX_JEZ_FILES_PER_PARTITION}, "
    f"colunas={len(FACT_URNA_LOGS_COLUMNS)})."
)

Configuração carregada para fact_urna_logs (ano=2026, turnos=[1, 2], UFs=['AC'], situacao='Totalizado', max_jez=5, colunas=15).


## 2. Inspeção do Central Directory, Extração em Memória de `.jez` (7z/LZMA e ZIP) e Ordenação de Logs Rotacionados

- Inspeciona o Central Directory do pacote `bu_imgbu_logjez_rdv_vscmr_2026_{turn}t_{uf}.zip` tanto a partir de arquivo local em `data/staging/election_year=2026/turn={turn}/uf={uf}/` quanto remotamente via HTTP Range.
- Filtra todas as entradas `o<pleito:5><uf:2><mun:5><zona:4><secao:4>-log.jez` e extrai `cd_pleito`, `cd_municipio`, `nr_zona`, `nr_secao` e `source_jez_file`.
- Detecta automaticamente os magic bytes do `.jez` (`7z\xbc\xaf\x27\x1c` para 7z/LZMA via `py7zr` e `PK\x03\x04` para ZIP via `zipfile`) e descompacta os membros em memória (`io.BytesIO`).
- Ordena deterministicamente os segmentos rotacionados (`logd.dat.2` -> `logd.dat.1` -> `logd.dat`).

In [2]:
import io
import re
import struct
import time
import zipfile
import zlib
from collections.abc import Sequence

import httpx
import py7zr
import py7zr.io

SEVEN_ZIP_MAGIC: bytes = b"7z\xbc\xaf\x27\x1c"
ZIP_MAGIC: bytes = b"PK\x03\x04"
MAX_IN_MEMORY_LOG_BYTES: int = 256 * 1024 * 1024

JEZ_ENTRY_RE = re.compile(
    r"^o(?P<pleito>\d{5})(?P<uf>[a-zA-Z]{2})(?P<mun>\d{5})(?P<zona>\d{4})(?P<secao>\d{4})-log\.jez$",
    re.IGNORECASE,
)
_ROTATED_LOG_RE = re.compile(r"^logd\.dat\.(\d+)(?:\.(?:jez|zip|7z))?$", re.IGNORECASE)
_CURRENT_LOG_RE = re.compile(r"^logd\.dat$", re.IGNORECASE)


def create_http_client(timeout: float = 120.0) -> httpx.Client:
    """Cria cliente HTTP configurado para requisições Range sem compressão intermediária."""
    return httpx.Client(
        timeout=timeout,
        follow_redirects=True,
        headers={"User-Agent": "tse-analysis/0.2", "Accept-Encoding": "identity"},
    )


def http_get_with_retry(
    client: httpx.Client,
    url: str,
    headers: dict[str, str] | None = None,
    max_retries: int = 4,
) -> httpx.Response:
    """Executa requisição GET com retry exponencial para resiliência contra falhas transitórias."""
    last_err: Exception | None = None
    for attempt in range(max_retries):
        try:
            resp = client.get(url, headers=headers)
            resp.raise_for_status()
            return resp
        except Exception as exc:
            last_err = exc
            if attempt + 1 < max_retries:
                time.sleep(1.5 * (2**attempt))
    raise RuntimeError(f"Falha após {max_retries} tentativas em {url}: {last_err}") from last_err


def download_file_to_staging(
    client: httpx.Client,
    url: str,
    dest_path: Path,
    max_retries: int = 4,
) -> Path:
    """Baixa pacote ZIP estadual em streaming para data/staging/ via arquivo .tmp + os.replace."""
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = dest_path.with_suffix(dest_path.suffix + ".tmp")
    last_err: Exception | None = None
    for attempt in range(max_retries):
        try:
            with client.stream("GET", url) as resp:
                resp.raise_for_status()
                with open(tmp_path, "wb") as fh:
                    for chunk in resp.iter_bytes(chunk_size=256 * 1024):
                        fh.write(chunk)
            os.replace(tmp_path, dest_path)
            return dest_path
        except Exception as exc:
            last_err = exc
            if tmp_path.exists():
                tmp_path.unlink(missing_ok=True)
            if attempt + 1 < max_retries:
                time.sleep(1.5 * (2**attempt))
    raise RuntimeError(f"Falha no download de {url}: {last_err}") from last_err


def parse_jez_entry_metadata(entry_name: str) -> dict | None:
    """Extrai cd_pleito, uf, cd_municipio, nr_zona, nr_secao e source_jez_file do nome do .jez."""
    basename = Path(entry_name).name
    match = JEZ_ENTRY_RE.match(basename)
    if match is None:
        return None
    return {
        "entry_name": entry_name,
        "source_jez_file": basename,
        "cd_pleito": int(match.group("pleito")),
        "uf": match.group("uf").upper(),
        "cd_municipio": int(match.group("mun")),
        "nr_zona": int(match.group("zona")),
        "nr_secao": int(match.group("secao")),
    }


def inspect_local_zip_central_directory(zip_source: Path | bytes | io.BytesIO) -> list[dict]:
    """Inspeciona o Central Directory de um ZIP local ou buffer em memória filtrando entradas *-log.jez."""
    target = io.BytesIO(zip_source) if isinstance(zip_source, bytes) else zip_source
    entries: list[dict] = []
    with zipfile.ZipFile(target, mode="r") as zf:
        for info in zf.infolist():
            if info.is_dir():
                continue
            meta = parse_jez_entry_metadata(info.filename)
            if meta is None:
                continue
            meta.update({
                "method": int(info.compress_type),
                "crc": int(info.CRC),
                "comp_size": int(info.compress_size),
                "uncomp_size": int(info.file_size),
                "lfh_offset": int(info.header_offset),
            })
            entries.append(meta)
    entries.sort(key=lambda item: item["source_jez_file"])
    return entries


def inspect_remote_zip_central_directory(client: httpx.Client, url: str) -> list[dict]:
    """Inspeciona o Central Directory de um pacote ZIP remoto no CDN do TSE via HTTP Range."""
    probe = http_get_with_retry(client, url, headers={"Range": "bytes=0-0"})
    content_range = probe.headers.get("content-range", "")
    if "/" in content_range:
        total_size = int(content_range.split("/")[-1])
    else:
        head_resp = client.head(url)
        head_resp.raise_for_status()
        total_size = int(head_resp.headers["content-length"])

    tail_len = min(total_size, 2 * 1024 * 1024)
    tail_resp = http_get_with_retry(
        client,
        url,
        headers={"Range": f"bytes={total_size - tail_len}-{total_size - 1}"},
    )
    tail_bytes = tail_resp.content
    eocd_pos = tail_bytes.rfind(b"PK\x05\x06")
    if eocd_pos < 0:
        raise ValueError(f"Assinatura EOCD (PK\\x05\\x06) não encontrada em {url}")

    cd_size, cd_offset = struct.unpack("<II", tail_bytes[eocd_pos + 12 : eocd_pos + 20])
    cd_resp = http_get_with_retry(
        client,
        url,
        headers={"Range": f"bytes={cd_offset}-{cd_offset + cd_size - 1}"},
    )
    cd_bytes = cd_resp.content

    pos = 0
    entries: list[dict] = []
    while pos + 46 <= len(cd_bytes):
        if cd_bytes[pos : pos + 4] != b"PK\x01\x02":
            break
        (
            _sig, _ver_made, _ver_need, _flags, method, _mtime, _mdate,
            crc, comp_size, uncomp_size, fn_len, extra_len, comment_len,
            _disk, _iattr, _eattr, lfh_offset,
        ) = struct.unpack("<4sHHHHHHIIIHHHHHII", cd_bytes[pos : pos + 46])
        filename = cd_bytes[pos + 46 : pos + 46 + fn_len].decode("latin1")
        pos += 46 + fn_len + extra_len + comment_len
        meta = parse_jez_entry_metadata(filename)
        if meta is not None:
            meta.update({
                "method": int(method),
                "crc": int(crc),
                "comp_size": int(comp_size),
                "uncomp_size": int(uncomp_size),
                "lfh_offset": int(lfh_offset),
            })
            entries.append(meta)

    entries.sort(key=lambda item: item["source_jez_file"])
    return entries


def fetch_remote_jez_entry_bytes(client: httpx.Client, url: str, entry: dict) -> bytes:
    """Baixa via HTTP Range e descomprime em memória os bytes de um único membro .jez do ZIP remoto."""
    lhoff = int(entry["lfh_offset"])
    comp_size = int(entry["comp_size"])
    end_byte = lhoff + 30 + len(entry["entry_name"]) + 256 + comp_size
    resp = http_get_with_retry(client, url, headers={"Range": f"bytes={lhoff}-{end_byte}"})
    buf = resp.content
    if buf[:4] != ZIP_MAGIC:
        raise ValueError(f"Local File Header inválido para {entry['source_jez_file']} em {url}")
    fn_len, extra_len = struct.unpack("<HH", buf[26:30])
    data_start = 30 + fn_len + extra_len
    comp_slice = buf[data_start : data_start + comp_size]

    method = int(entry["method"])
    if method == 0:
        raw_bytes = comp_slice
    elif method == 8:
        raw_bytes = zlib.decompress(comp_slice, -zlib.MAX_WBITS)
    else:
        raise ValueError(f"Método de compressão ZIP não suportado: {method}")

    expected_size = int(entry["uncomp_size"])
    if len(raw_bytes) != expected_size:
        raise ValueError(
            f"Tamanho descomprimido divergente em {entry['source_jez_file']}: "
            f"{len(raw_bytes)} != {expected_size}"
        )
    actual_crc = zlib.crc32(raw_bytes) & 0xFFFFFFFF
    expected_crc = int(entry["crc"]) & 0xFFFFFFFF
    if actual_crc != expected_crc:
        raise ValueError(
            f"CRC32 divergente em {entry['source_jez_file']}: {actual_crc:#x} != {expected_crc:#x}"
        )
    return raw_bytes


def detect_jez_container_format(jez_bytes: bytes) -> str:
    """Detecta automaticamente a assinatura mágica do contêiner .jez: 7z/LZMA ou ZIP."""
    if jez_bytes.startswith(SEVEN_ZIP_MAGIC):
        return "7z"
    if jez_bytes.startswith(ZIP_MAGIC):
        return "zip"
    raise ValueError(
        "Assinatura mágica desconhecida no arquivo .jez: esperado 7z/LZMA "
        "(37 7A BC AF 27 1C) ou ZIP (50 4B 03 04)."
    )


def _classify_log_member_rotation(member_name: str) -> int | None:
    """Retorna índice de rotação: 0 para logd.dat ativo, >=1 para logd.dat.N ou None para outros arquivos."""
    basename = Path(member_name).name.strip()
    if _CURRENT_LOG_RE.fullmatch(basename):
        return 0
    m_rot = _ROTATED_LOG_RE.fullmatch(basename)
    if m_rot is not None:
        return int(m_rot.group(1))
    return None


def sort_rotated_log_members(member_names: Sequence[str]) -> list[str]:
    """Ordena deterministicamente os segmentos de log do mais antigo ao atual: logd.dat.2 -> logd.dat.1 -> logd.dat."""
    classified: list[tuple[int, str]] = []
    for name in member_names:
        rot_idx = _classify_log_member_rotation(name)
        if rot_idx is not None:
            classified.append((rot_idx, name))
    classified.sort(key=lambda item: (-item[0], item[1]))
    return [name for _, name in classified]


def _extract_raw_archive_in_memory(jez_bytes: bytes) -> tuple[str, dict[str, bytes]]:
    """Descompacta os membros de um contêiner 7z/LZMA ou ZIP inteiramente em memória (io.BytesIO)."""
    fmt = detect_jez_container_format(jez_bytes)
    buf = io.BytesIO(jez_bytes)
    if fmt == "7z":
        with py7zr.SevenZipFile(buf, mode="r") as archive:
            if hasattr(py7zr.io, "BytesIOFactory"):
                factory = py7zr.io.BytesIOFactory(limit=MAX_IN_MEMORY_LOG_BYTES)
                archive.extract(factory=factory)
                members: dict[str, bytes] = {}
                for name, bio in factory.products.items():
                    bio.seek(0)
                    members[name] = bio.read()
                return fmt, members
            read_fn = getattr(archive, "read", None) or getattr(archive, "readall", None)
            if read_fn is not None:
                raw_map = read_fn()
                return fmt, {name: bio.read() for name, bio in raw_map.items()}
            raise RuntimeError("Versão de py7zr sem suporte a extração em memória.")

    members_zip: dict[str, bytes] = {}
    with zipfile.ZipFile(buf, mode="r") as zf:
        for info in zf.infolist():
            if info.is_dir():
                continue
            members_zip[info.filename] = zf.read(info.filename)
    return fmt, members_zip


def extract_jez_log_members_in_memory(
    jez_bytes: bytes,
    _depth: int = 0,
) -> tuple[str, list[tuple[str, bytes]]]:
    """Extrai em memória todos os segmentos logd.dat* de um .jez ordenados cronologicamente."""
    fmt, raw_members = _extract_raw_archive_in_memory(jez_bytes)
    flattened: dict[str, bytes] = {}
    for member_name, payload in raw_members.items():
        if (
            _depth < 2
            and (payload.startswith(SEVEN_ZIP_MAGIC) or payload.startswith(ZIP_MAGIC))
        ):
            _, nested_items = extract_jez_log_members_in_memory(payload, _depth=_depth + 1)
            parent_rot = _classify_log_member_rotation(member_name)
            for inner_name, inner_bytes in nested_items:
                inner_rot = _classify_log_member_rotation(inner_name)
                if parent_rot is not None and parent_rot > 0 and inner_rot == 0:
                    synthetic_name = f"logd.dat.{parent_rot}"
                else:
                    synthetic_name = Path(inner_name).name
                flattened[synthetic_name] = inner_bytes
        else:
            flattened[Path(member_name).name] = payload

    ordered_names = sort_rotated_log_members(list(flattened.keys()))
    return fmt, [(name, flattened[name]) for name in ordered_names]

## 3. Parser 1:1 para `fact_urna_logs` (15 Colunas), Mascaramento `<ID_REDACTED>`, Escrita Atômica Parquet ZSTD e Checkpointing

- Decodifica cada linha física de `logd.dat` (`utf-8` estrito com fallback `latin1`) preservando 100% das linhas físicas.
- Aplica `re.sub(r"\b\d{5,}\b", "<ID_REDACTED>", msg)` em `message_text` e omite o 6º campo hexadecimal `mac_code`.
- Grava de forma atômica (via `.parquet.tmp` + `os.replace`) em `data/exports/fact_urna_logs/election_year=2026/turn={turn}/uf={uf}/part-0.parquet` com `compression='zstd'`, `compression_level=5` e `use_dictionary=True`.
- Valida `processed_jez_count == target_jez_count` e `physical_logd_lines_count == parquet_rows_count`, limpa imediatamente `data/staging/` e atualiza atomicamente `data/exports/_checkpoints/manifest.json`.

In [3]:
import gc
import json
import shutil

import pyarrow.parquet as pq

ID_REDACTION_RE = re.compile(r"\b\d{5,}\b")
_NON_ALPHA_ID_REDACTION_RE = re.compile(r"(?<![A-Za-z\d])\d{5,}(?![A-Za-z\d])")
_MAC_HEX_RE = re.compile(r"^[0-9A-Fa-f]{8,64}$")
_DATE_ONLY_RE = re.compile(r"^\d{2}/\d{2}/\d{4}$")
_TIME_ONLY_RE = re.compile(r"^\d{2}:\d{2}:\d{2}(?:[.,]\d+)?$")


def redact_message_text(msg: str) -> str:
    """Mascara sequências numéricas de 5 ou mais dígitos como <ID_REDACTED> em message_text."""
    redacted = ID_REDACTION_RE.sub("<ID_REDACTED>", msg.strip())
    return _NON_ALPHA_ID_REDACTION_RE.sub("<ID_REDACTED>", redacted)


def decode_logd_line(raw_line: bytes | str) -> str:
    """Decodifica linha de logd.dat tentando UTF-8 estrito com fallback para ISO-8859-1 (latin1)."""
    if isinstance(raw_line, str):
        return raw_line.rstrip("\r\n")
    try:
        return raw_line.decode("utf-8-sig", errors="strict").rstrip("\r\n")
    except UnicodeDecodeError:
        return raw_line.decode("latin1").rstrip("\r\n")


def parse_logd_tab_line(raw_line: bytes | str) -> tuple[str, str, str, str, str]:
    """Converte 1:1 uma linha tabulada de logd.dat em (raw_timestamp, log_level, urna_serial, application, message_text).

    Omite o 6º campo hexadecimal (mac_code) e aplica mascaramento <ID_REDACTED> em message_text,
    sem inferir event_code nem descartar linhas físicas.
    """
    line_str = decode_logd_line(raw_line)
    parts = line_str.split("\t")

    # Suporta formato de 6 campos (DD/MM/YYYY HH:MM:SS \t LEVEL \t SERIAL \t APP \t MSG \t MAC)
    # e variante de 7 campos onde data e hora vêm separadas por tabulação
    if (
        len(parts) >= 6
        and _DATE_ONLY_RE.fullmatch(parts[0].strip())
        and _TIME_ONLY_RE.fullmatch(parts[1].strip())
    ):
        raw_ts = f"{parts[0].strip()} {parts[1].strip()}"
        log_level = parts[2].strip()
        urna_serial = parts[3].strip()
        application = parts[4].strip()
        tail_fields = parts[5:]
    elif len(parts) >= 5:
        raw_ts = parts[0].strip()
        log_level = parts[1].strip()
        urna_serial = parts[2].strip()
        application = parts[3].strip()
        tail_fields = parts[4:]
    else:
        # Preserva linha física atípica/truncada sem perda de contagem 1:1 e sem vazar mac_code final
        clean_parts = (
            parts[:-1]
            if len(parts) >= 2 and _MAC_HEX_RE.fullmatch(parts[-1].strip())
            else parts
        )
        raw_ts = redact_message_text(clean_parts[0]) if len(clean_parts) > 0 else ""
        log_level = redact_message_text(clean_parts[1]) if len(clean_parts) > 1 else "UNKNOWN"
        urna_serial = "UNKNOWN"
        application = "UNKNOWN"
        fallback_msg = "\t".join(clean_parts)
        return (raw_ts, log_level, urna_serial, application, redact_message_text(fallback_msg))

    # Omite o 6º campo tabulado (mac_code) quando presente ao final da linha,
    # inclusive em linhas de 5/6 campos onde a coluna APP foi omitida antes de MSG\tMAC
    if len(tail_fields) >= 2:
        msg_raw = "\t".join(tail_fields[:-1])
    elif len(tail_fields) == 1:
        if _MAC_HEX_RE.fullmatch(tail_fields[0].strip()):
            msg_raw = application
            application = "SYSTEM"
        else:
            msg_raw = tail_fields[0]
    else:
        msg_raw = ""

    return (raw_ts, log_level, urna_serial, application, redact_message_text(msg_raw))


def parse_jez_bytes_to_fact_rows(
    jez_bytes: bytes,
    jez_meta: dict,
    election_year: int,
    turn: int,
) -> tuple[list[dict], int, str, list[str]]:
    """Extrai um arquivo .jez em memória e gera os registros de 15 colunas de fact_urna_logs com file_seq 1..N."""
    container_fmt, ordered_members = extract_jez_log_members_in_memory(jez_bytes)
    rows: list[dict] = []
    physical_lines = 0
    file_seq = 1

    uf_str = str(jez_meta["uf"]).upper()
    cd_pleito = int(jez_meta["cd_pleito"])
    cd_municipio = int(jez_meta["cd_municipio"])
    nr_zona = int(jez_meta["nr_zona"])
    nr_secao = int(jez_meta["nr_secao"])
    source_jez_file = str(jez_meta["source_jez_file"])

    for member_name, member_bytes in ordered_members:
        for raw_line in member_bytes.splitlines():
            if not raw_line.strip():
                continue
            physical_lines += 1
            raw_ts, log_level, urna_serial, application, message_text = parse_logd_tab_line(
                raw_line
            )
            rows.append({
                "election_year": int(election_year),
                "turn": int(turn),
                "uf": uf_str,
                "cd_pleito": cd_pleito,
                "cd_municipio": cd_municipio,
                "nr_zona": nr_zona,
                "nr_secao": nr_secao,
                "source_jez_file": source_jez_file,
                "source_log_member": member_name,
                "file_seq": file_seq,
                "raw_timestamp": raw_ts,
                "log_level": log_level,
                "urna_serial": urna_serial,
                "application": application,
                "message_text": message_text,
            })
            file_seq += 1

    member_names = [m_name for m_name, _ in ordered_members]
    return rows, physical_lines, container_fmt, member_names


def partition_output_path(
    exports_dir: Path,
    election_year: int,
    turn: int,
    uf: str,
) -> Path:
    """Constrói o caminho Hive canônico data/exports/fact_urna_logs/election_year=YYYY/turn=T/uf=UF/part-0.parquet."""
    return (
        exports_dir
        / "fact_urna_logs"
        / f"election_year={election_year}"
        / f"turn={turn}"
        / f"uf={uf.upper()}"
        / "part-0.parquet"
    )


def write_fact_urna_logs_parquet_atomic(
    df_or_batches: pl.DataFrame | list[pl.DataFrame],
    target_path: Path,
) -> tuple[int, int]:
    """Escreve partição fact_urna_logs em Parquet ZSTD (nível 5, dictionary encoding) via arquivo .tmp + os.replace."""
    target_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = target_path.with_suffix(".parquet.tmp")
    dfs = [df_or_batches] if isinstance(df_or_batches, pl.DataFrame) else list(df_or_batches)
    try:
        with pq.ParquetWriter(
            tmp_path,
            schema=FACT_URNA_LOGS_ARROW_SCHEMA,
            compression="zstd",
            compression_level=5,
            use_dictionary=True,
            write_statistics=True,
        ) as writer:
            for chunk_df in dfs:
                if chunk_df.height == 0:
                    continue
                aligned_df = chunk_df.select(FACT_URNA_LOGS_COLUMNS).cast(FACT_URNA_LOGS_SCHEMA)
                arrow_tbl = aligned_df.to_arrow().cast(FACT_URNA_LOGS_ARROW_SCHEMA)
                writer.write_table(arrow_tbl)
        os.replace(tmp_path, target_path)
    finally:
        if tmp_path.exists():
            tmp_path.unlink(missing_ok=True)

    meta = pq.read_metadata(target_path)
    return int(meta.num_rows), int(target_path.stat().st_size)


def load_manifest(manifest_path: Path) -> dict:
    """Carrega o manifesto incremental de checkpoints ou inicializa estrutura vazia."""
    if manifest_path.exists():
        try:
            return json.loads(manifest_path.read_text(encoding="utf-8"))
        except Exception:
            pass
    return {
        "schema_version": "1.0",
        "election_year": ELECTION_YEAR,
        "partitions": {},
        "tables_summary": {},
    }


def save_manifest_atomic(manifest_path: Path, manifest_data: dict) -> None:
    """Persiste o manifesto incremental de forma atômica via arquivo .tmp + os.replace."""
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = manifest_path.with_suffix(".json.tmp")
    try:
        tmp_path.write_text(
            json.dumps(manifest_data, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )
        os.replace(tmp_path, manifest_path)
    finally:
        if tmp_path.exists():
            tmp_path.unlink(missing_ok=True)


def clean_staging_dir(staging_dir: Path) -> None:
    """Remove imediatamente todos os artefatos e subdiretórios temporários de data/staging/."""
    if staging_dir.exists():
        for item in staging_dir.iterdir():
            if item.is_dir():
                shutil.rmtree(item, ignore_errors=True)
            else:
                item.unlink(missing_ok=True)


def clean_partition_staging(part_staging_dir: Path, staging_dir: Path) -> None:
    """Remove imediatamente o subdiretório temporário da partição e poda diretórios pais vazios."""
    if part_staging_dir.exists():
        shutil.rmtree(part_staging_dir, ignore_errors=True)
    if staging_dir.exists():
        for item in list(staging_dir.iterdir()):
            if item.is_file():
                item.unlink(missing_ok=True)
        cur = part_staging_dir.parent
        while cur != staging_dir and cur.exists():
            try:
                cur.rmdir()
            except OSError:
                break
            cur = cur.parent


def ingest_uf_turn_partition(
    election_year: int,
    turn: int,
    uf: str,
    staging_dir: Path = STAGING_DIR,
    exports_dir: Path = EXPORTS_DIR,
    manifest_path: Path = MANIFEST_PATH,
    url_template: str = URL_TEMPLATE,
    max_jez_files: int | None = MAX_JEZ_FILES_PER_PARTITION,
    skip_existing: bool = SKIP_EXISTING,
    force_refresh: bool = FORCE_REFRESH,
    local_zip_path: Path | None = None,
    download_full_zip: bool = False,
    chunk_jez_size: int = 100,
    client: httpx.Client | None = None,
) -> dict:
    """Processa de ponta a ponta uma partição (election_year, turn, uf) de fact_urna_logs com reconciliação física."""
    uf_upper = uf.strip().upper()
    part_key = f"fact_urna_logs/election_year={election_year}/turn={turn}/uf={uf_upper}"
    out_path = partition_output_path(exports_dir, election_year, turn, uf_upper)
    zip_url = url_template.format(election_year=election_year, turn=turn, uf=uf_upper)
    part_staging_dir = (
        staging_dir / f"election_year={election_year}" / f"turn={turn}" / f"uf={uf_upper}"
    )

    manifest = load_manifest(manifest_path)
    existing_rec = manifest.get("partitions", {}).get(part_key, {})
    if (
        skip_existing
        and not force_refresh
        and out_path.exists()
        and existing_rec.get("status") == "completed"
        and existing_rec.get("max_jez_files") == max_jez_files
    ):
        cached_df = pl.read_parquet(out_path)
        assert cached_df.width == 15
        assert list(cached_df.columns) == FACT_URNA_LOGS_COLUMNS
        assert cached_df.height == int(existing_rec["parquet_row_count"])
        assert int(existing_rec["processed_jez_count"]) == int(existing_rec["target_jez_count"])
        assert int(existing_rec["physical_logd_lines_count"]) == int(
            existing_rec["parquet_row_count"]
        )
        clean_partition_staging(part_staging_dir, staging_dir)
        return {
            **existing_rec,
            "head_df": cached_df.head(5),
            "tail_df": cached_df.tail(5),
            "cached": True,
        }

    own_client = client is None
    http_client = client or create_http_client()
    part_staging_dir.mkdir(parents=True, exist_ok=True)
    staged_zip_candidate = (
        local_zip_path
        if local_zip_path is not None
        else part_staging_dir / f"bu_imgbu_logjez_rdv_vscmr_{election_year}_{turn}t_{uf_upper}.zip"
    )

    try:
        if staged_zip_candidate is not None and staged_zip_candidate.exists():
            all_cd_entries = inspect_local_zip_central_directory(staged_zip_candidate)
            use_local_zip = True
        elif download_full_zip:
            download_file_to_staging(http_client, zip_url, staged_zip_candidate)
            all_cd_entries = inspect_local_zip_central_directory(staged_zip_candidate)
            use_local_zip = True
        else:
            all_cd_entries = inspect_remote_zip_central_directory(http_client, zip_url)
            use_local_zip = False

        central_directory_jez_count = len(all_cd_entries)
        target_entries = (
            all_cd_entries[:max_jez_files] if max_jez_files is not None else all_cd_entries
        )
        target_jez_count = len(target_entries)
        if target_jez_count == 0:
            raise ValueError(f"Nenhuma entrada .jez encontrada para {part_key} ({zip_url})")

        processed_jez_count = 0
        physical_logd_lines_count = 0
        detected_formats: set[str] = set()
        chunk_dfs: list[pl.DataFrame] = []
        accumulated_rows: list[dict] = []

        if use_local_zip:
            with zipfile.ZipFile(staged_zip_candidate, mode="r") as zf:
                for entry in target_entries:
                    jez_bytes = zf.read(entry["entry_name"])
                    rows, phys_lines, fmt, _members = parse_jez_bytes_to_fact_rows(
                        jez_bytes, entry, election_year=election_year, turn=turn
                    )
                    detected_formats.add(fmt)
                    processed_jez_count += 1
                    physical_logd_lines_count += phys_lines
                    accumulated_rows.extend(rows)
                    if len(accumulated_rows) >= chunk_jez_size * 4000:
                        chunk_dfs.append(
                            pl.DataFrame(accumulated_rows, schema=FACT_URNA_LOGS_SCHEMA)
                        )
                        accumulated_rows = []
        else:
            for entry in target_entries:
                jez_bytes = fetch_remote_jez_entry_bytes(http_client, zip_url, entry)
                rows, phys_lines, fmt, _members = parse_jez_bytes_to_fact_rows(
                    jez_bytes, entry, election_year=election_year, turn=turn
                )
                detected_formats.add(fmt)
                processed_jez_count += 1
                physical_logd_lines_count += phys_lines
                accumulated_rows.extend(rows)
                if len(accumulated_rows) >= chunk_jez_size * 4000:
                    chunk_dfs.append(
                        pl.DataFrame(accumulated_rows, schema=FACT_URNA_LOGS_SCHEMA)
                    )
                    accumulated_rows = []

        if accumulated_rows or not chunk_dfs:
            chunk_dfs.append(pl.DataFrame(accumulated_rows, schema=FACT_URNA_LOGS_SCHEMA))

        # Validação obrigatória pós-extração e pós-escrita
        if processed_jez_count != target_jez_count:
            raise ValueError(
                f"Divergência de arquivos .jez em {part_key}: "
                f"processed={processed_jez_count} != target={target_jez_count}"
            )

        parquet_rows_count, size_bytes = write_fact_urna_logs_parquet_atomic(chunk_dfs, out_path)
        if physical_logd_lines_count != parquet_rows_count:
            raise ValueError(
                f"Divergência de linhas físicas em {part_key}: "
                f"physical_logd={physical_logd_lines_count} != parquet={parquet_rows_count}"
            )

        reloaded_df = pl.read_parquet(out_path)
        if reloaded_df.width != 15 or list(reloaded_df.columns) != FACT_URNA_LOGS_COLUMNS:
            raise ValueError(
                f"Schema inválido em {out_path}: colunas={reloaded_df.columns}"
            )
        if reloaded_df.height != physical_logd_lines_count:
            raise ValueError(
                f"Contagem lida do Parquet ({reloaded_df.height}) != "
                f"linhas físicas ({physical_logd_lines_count})"
            )

        part_record = {
            "table_name": "fact_urna_logs",
            "election_year": int(election_year),
            "turn": int(turn),
            "uf": uf_upper,
            "situacao": REQUIRED_SITUACAO,
            "source_url": zip_url,
            "container_formats": sorted(detected_formats),
            "central_directory_jez_count": central_directory_jez_count,
            "target_jez_count": target_jez_count,
            "processed_jez_count": processed_jez_count,
            "max_jez_files": max_jez_files,
            "column_count": 15,
            "physical_logd_lines_count": physical_logd_lines_count,
            "parquet_row_count": parquet_rows_count,
            "size_bytes": size_bytes,
            "parquet_path": str(out_path),
            "status": "completed",
        }
        manifest = load_manifest(manifest_path)
        manifest.setdefault("partitions", {})[part_key] = part_record
        save_manifest_atomic(manifest_path, manifest)
        head_slice = reloaded_df.head(5)
        tail_slice = reloaded_df.tail(5)
        del chunk_dfs, reloaded_df, accumulated_rows
        gc.collect()
        return {
            **part_record,
            "head_df": head_slice,
            "tail_df": tail_slice,
            "cached": False,
        }
    finally:
        clean_partition_staging(part_staging_dir, staging_dir)
        if own_client:
            http_client.close()


def ingest_urna_logs_batch(
    election_year: int = ELECTION_YEAR,
    turns: list[int] = TARGET_TURNS,
    target_ufs: list[str] = TARGET_UFS,
    staging_dir: Path = STAGING_DIR,
    exports_dir: Path = EXPORTS_DIR,
    manifest_path: Path = MANIFEST_PATH,
    url_template: str = URL_TEMPLATE,
    max_jez_files: int | None = MAX_JEZ_FILES_PER_PARTITION,
    skip_existing: bool = SKIP_EXISTING,
    force_refresh: bool = FORCE_REFRESH,
    client: httpx.Client | None = None,
) -> dict:
    """Orquestra a ingestão em batch de fact_urna_logs para múltiplos turnos e UFs e consolida o manifesto."""
    own_client = client is None
    http_client = client or create_http_client()
    partitions_results: list[dict] = []
    head_df: pl.DataFrame | None = None
    tail_df: pl.DataFrame | None = None
    try:
        for turn in turns:
            for uf in target_ufs:
                res = ingest_uf_turn_partition(
                    election_year=election_year,
                    turn=turn,
                    uf=uf,
                    staging_dir=staging_dir,
                    exports_dir=exports_dir,
                    manifest_path=manifest_path,
                    url_template=url_template,
                    max_jez_files=max_jez_files,
                    skip_existing=skip_existing,
                    force_refresh=force_refresh,
                    client=http_client,
                )
                partitions_results.append(res)
                if head_df is None:
                    head_df = res["head_df"]
                tail_df = res["tail_df"]

        if head_df is None:
            head_df = pl.DataFrame(schema=FACT_URNA_LOGS_SCHEMA)
        if tail_df is None:
            tail_df = pl.DataFrame(schema=FACT_URNA_LOGS_SCHEMA)
        total_phys_lines = sum(int(p["physical_logd_lines_count"]) for p in partitions_results)
        total_pq_rows = sum(int(p["parquet_row_count"]) for p in partitions_results)
        total_jez_processed = sum(int(p["processed_jez_count"]) for p in partitions_results)
        total_jez_target = sum(int(p["target_jez_count"]) for p in partitions_results)
        total_size_bytes = sum(int(p["size_bytes"]) for p in partitions_results)

        if total_jez_processed != total_jez_target:
            raise ValueError(
                f"Reconciliação total de .jez falhou: {total_jez_processed} != {total_jez_target}"
            )
        if total_phys_lines != total_pq_rows:
            raise ValueError(
                f"Reconciliação total de linhas falhou: {total_phys_lines} != {total_pq_rows}"
            )

        table_summary = {
            "table_name": "fact_urna_logs",
            "column_count": 15,
            "expected_cols": 15,
            "partitions_count": len(partitions_results),
            "target_jez_count": total_jez_target,
            "processed_jez_count": total_jez_processed,
            "physical_logd_lines_count": total_phys_lines,
            "parquet_row_count": total_pq_rows,
            "total_size_bytes": total_size_bytes,
        }
        manifest = load_manifest(manifest_path)
        manifest.setdefault("tables_summary", {})["fact_urna_logs"] = table_summary
        save_manifest_atomic(manifest_path, manifest)

        return {
            **table_summary,
            "partitions": partitions_results,
            "head_df": head_df,
            "tail_df": tail_df,
        }
    finally:
        clean_staging_dir(staging_dir)
        if own_client:
            http_client.close()


def compute_parquet_storage_report(exports_dir: Path) -> tuple[list[dict], int]:
    """Calcula o tamanho em disco (bytes, KB, MB) de CADA arquivo Parquet ZSTD em data/exports/ e a soma total."""
    files_report: list[dict] = []
    if not exports_dir.exists():
        return files_report, 0

    for pq_file in sorted(exports_dir.rglob("*.parquet")):
        size_bytes = int(pq_file.stat().st_size)
        meta = pq.read_metadata(pq_file)
        rel_path = pq_file.relative_to(exports_dir)
        table_name = rel_path.parts[0] if rel_path.parts else pq_file.name
        files_report.append({
            "table_name": table_name,
            "parquet_path": str(pq_file),
            "relative_path": str(rel_path),
            "row_count": int(meta.num_rows),
            "column_count": int(meta.num_columns),
            "size_bytes": size_bytes,
            "size_kb": round(size_bytes / 1024.0, 2),
            "size_mb": round(size_bytes / (1024.0 * 1024.0), 4),
        })
    total_bytes = sum(item["size_bytes"] for item in files_report)
    return files_report, total_bytes


def prepare_kaggle_dataset_incremental_update(
    exports_dir: Path = EXPORTS_DIR,
    dataset_slug: str = "daniellmattos/tse-analysis-eleicoes-2026",
    execute_push: bool = False,
) -> dict:
    """Prepara metadados e (opcionalmente) publica atualização incremental das partições validadas no Kaggle Dataset."""
    exports_dir.mkdir(parents=True, exist_ok=True)
    meta_path = exports_dir / "dataset-metadata.json"
    tmp_meta_path = meta_path.with_suffix(".json.tmp")
    dataset_meta = {
        "title": "TSE Analysis Eleicoes 2026 Parquet ZSTD",
        "id": dataset_slug,
        "licenses": [{"name": "CC-BY-4.0"}],
    }
    try:
        tmp_meta_path.write_text(
            json.dumps(dataset_meta, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        os.replace(tmp_meta_path, meta_path)
    finally:
        if tmp_meta_path.exists():
            tmp_meta_path.unlink(missing_ok=True)
    urna_parquets = sorted((exports_dir / "fact_urna_logs").rglob("*.parquet")) if (
        exports_dir / "fact_urna_logs"
    ).exists() else []
    status = "ready_for_incremental_publish"
    if execute_push and shutil.which("kaggle"):
        subprocess.run(
            [
                "kaggle",
                "datasets",
                "version",
                "-p",
                str(exports_dir),
                "-m",
                "Incremental append fact_urna_logs partitions",
                "--dir-mode",
                "zip",
            ],
            check=True,
        )
        status = "published"
    return {
        "dataset_slug": dataset_slug,
        "metadata_path": str(meta_path),
        "fact_urna_logs_partitions": len(urna_parquets),
        "status": status,
    }

## 4. Execução em Batch (`TARGET_TURNS = [1, 2]`, `TARGET_UFS = ['AC']`), Inspeção `HEAD(5)` / `TAIL(5)` e Relatório de Tamanho em Disco Parquet ZSTD

In [4]:
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(10)
pl.Config.set_tbl_width_chars(240)
pl.Config.set_fmt_str_lengths(90)

batch_summary = ingest_urna_logs_batch(
    election_year=ELECTION_YEAR,
    turns=TARGET_TURNS,
    target_ufs=TARGET_UFS,
    staging_dir=STAGING_DIR,
    exports_dir=EXPORTS_DIR,
    manifest_path=MANIFEST_PATH,
    url_template=URL_TEMPLATE,
    max_jez_files=MAX_JEZ_FILES_PER_PARTITION,
    skip_existing=SKIP_EXISTING,
    force_refresh=FORCE_REFRESH,
)

print("=" * 110)
print(
    f"TABELA: fact_urna_logs | Colunas: {batch_summary['column_count']}/{batch_summary['expected_cols']} "
    f"| Partições: {batch_summary['partitions_count']} "
    f"| .jez processados == alvo: {batch_summary['processed_jez_count']} == {batch_summary['target_jez_count']} "
    f"| Linhas logd.dat == Parquet: {batch_summary['physical_logd_lines_count']:,} == {batch_summary['parquet_row_count']:,} "
    f"| Disco fact_urna_logs: {batch_summary['total_size_bytes']:,} bytes ({batch_summary['total_size_bytes']/1024:.2f} KB / {batch_summary['total_size_bytes']/(1024*1024):.4f} MB)"
)
print("=" * 110)

# a) HEAD(5) e TAIL(5) de cada partição gerada de fact_urna_logs
for part in batch_summary["partitions"]:
    part_kb = part["size_bytes"] / 1024.0
    part_mb = part["size_bytes"] / (1024.0 * 1024.0)
    print(
        f"\n-> Partição election_year={part['election_year']}/turn={part['turn']}/uf={part['uf']} "
        f"| cd_jez={part['central_directory_jez_count']:,} "
        f"| jez_processados={part['processed_jez_count']}/{part['target_jez_count']} "
        f"| linhas_logd==parquet: {part['physical_logd_lines_count']:,} == {part['parquet_row_count']:,} "
        f"| size={part['size_bytes']:,} bytes ({part_kb:.2f} KB / {part_mb:.4f} MB) "
        f"| path={part['parquet_path']}"
    )
    print(f"--- fact_urna_logs (turn={part['turn']}, uf={part['uf']}) : HEAD(5) ---")
    print(part["head_df"])
    print(f"--- fact_urna_logs (turn={part['turn']}, uf={part['uf']}) : TAIL(5) ---")
    print(part["tail_df"])

print("\n--- fact_urna_logs : HEAD(5) ---")
print(batch_summary["head_df"])
print("\n--- fact_urna_logs : TAIL(5) ---")
print(batch_summary["tail_df"])
print()

# b) Tamanho em disco (em bytes e KB/MB) de CADA arquivo Parquet ZSTD gerado e c) SOMA TOTAL em disco
files_report, total_disk_bytes = compute_parquet_storage_report(EXPORTS_DIR)
urna_files_report = [f for f in files_report if f["table_name"] == "fact_urna_logs"]
urna_total_bytes = sum(f["size_bytes"] for f in urna_files_report)
distinct_tables_count = len({f["table_name"] for f in files_report})

print("#" * 110)
print("RELATÓRIO DE ARMAZENAMENTO PARQUET ZSTD POR ARQUIVO GERADO")
print("#" * 110)
for i, f_info in enumerate(files_report, start=1):
    print(
        f"{i:02d}. {f_info['parquet_path']} | "
        f"rows={f_info['row_count']:,} | cols={f_info['column_count']} | "
        f"size={f_info['size_bytes']:,} bytes ({f_info['size_kb']:.2f} KB / {f_info['size_mb']:.4f} MB)"
    )

urna_kb = urna_total_bytes / 1024.0
urna_mb = urna_total_bytes / (1024.0 * 1024.0)
total_disk_kb = total_disk_bytes / 1024.0
total_disk_mb = total_disk_bytes / (1024.0 * 1024.0)
print("-" * 110)
print(
    f"SUBTOTAL EM DISCO fact_urna_logs ({len(urna_files_report)} arquivos Parquet ZSTD): "
    f"{urna_total_bytes:,} bytes ({urna_kb:,.2f} KB / {urna_mb:.4f} MB)"
)
print(
    f"SOMA TOTAL EM DISCO ({len(files_report)} arquivos Parquet ZSTD, {distinct_tables_count} tabelas): "
    f"{total_disk_bytes:,} bytes ({total_disk_kb:,.2f} KB / {total_disk_mb:.4f} MB)"
)
print("#" * 110)

kaggle_sync_info = prepare_kaggle_dataset_incremental_update(EXPORTS_DIR, execute_push=False)
print(
    f"Integração Kaggle Dataset: status={kaggle_sync_info['status']} | "
    f"partições fact_urna_logs={kaggle_sync_info['fact_urna_logs_partitions']} | "
    f"metadata={kaggle_sync_info['metadata_path']}"
)

# Verificações finais de integridade física, limpeza de staging e checkpoint
assert batch_summary["processed_jez_count"] == batch_summary["target_jez_count"]
assert batch_summary["physical_logd_lines_count"] == batch_summary["parquet_row_count"]
staging_leftovers = list(STAGING_DIR.glob("*")) if STAGING_DIR.exists() else []
assert len(staging_leftovers) == 0, f"Diretório staging não está vazio: {staging_leftovers}"
assert MANIFEST_PATH.exists(), "Arquivo manifest.json não foi gerado"
print(
    "Auditoria concluída com 100% de sucesso: fact_urna_logs reconciliado 1:1, "
    "staging limpo e manifest.json atualizado."
)

TABELA: fact_urna_logs | Colunas: 15/15 | Partições: 2 | .jez processados == alvo: 10 == 10 | Linhas logd.dat == Parquet: 73,130 == 73,130 | Disco fact_urna_logs: 426,800 bytes (416.80 KB / 0.4070 MB)

-> Partição election_year=2026/turn=1/uf=AC | cd_jez=2,270 | jez_processados=5/5 | linhas_logd==parquet: 28,923 == 28,923 | size=160,640 bytes (156.88 KB / 0.1532 MB) | path=data/exports/fact_urna_logs/election_year=2026/turn=1/uf=AC/part-0.parquet
--- fact_urna_logs (turn=1, uf=AC) : HEAD(5) ---
shape: (5, 15)
┌───────────────┬──────┬─────┬───────────┬──────────────┬─────────┬──────────┬───────────────────────────────┬───────────────────┬──────────┬─────────────────────┬───────────┬─────────────┬─────────────┬────────────────────────────────────┐
│ election_year ┆ turn ┆ uf  ┆ cd_pleito ┆ cd_municipio ┆ nr_zona ┆ nr_secao ┆ source_jez_file               ┆ source_log_member ┆ file_seq ┆ raw_timestamp       ┆ log_level ┆ urna_serial ┆ application ┆ message_text                       │
│ -